# Session 4 — Exceptions, Files & Research Data

> try/except, raising, validating dirty input; open/with, CSV, statistics, the pandas teaser.

**How to use this notebook:** read each cell, **predict** what it prints, then run it with **Shift + Enter**. Change one thing and predict again — the surprise is the lesson. Practice tasks (with collapsed solutions) are at the bottom.

**Tips:** press **Tab** to autocomplete a name, and **Shift + Tab** for a function's help. Need a library? Run `%pip install <name>` in a cell (e.g. `%pip install pandas`) — in the browser (JupyterLite) that fetches a Pyodide build and lasts for the session.

In [ ]:
# Setup (notebook only): write the data files Part B of this session reads.
from pathlib import Path
Path('students.csv').write_text('name,major,score\nAna,Education,91\nBen,Psychology,58\nCara,Education,73\nDev,Sociology,64\nEve,Psychology,88\nFinn,Education,79\n')
Path('survey.csv').write_text('respondent,q1_engagement,q2_clarity,q3_workload,q4_support\nR01,5,4,2,5\nR02,4,4,3,4\nR03,3,5,N/A,4\nR04,5,5,1,5\nR05,2,3,4,2\nR06,4,,3,4\nR07,5,4,2,5\nR08,1,2,5,1\n')
print('wrote students.csv, survey.csv')

In [ ]:
# ======================================================================
# PART A — Exceptions & Defensive Code

In [ ]:
# ======================================================================

In [ ]:
# --- 1. try / except: convert what you can ------------------------------
def safe_int(value):
    """Return int(value) or None if it can't be parsed."""
    try:
        return int(value)
    except (ValueError, TypeError):
        return None

for v in ["42", "N/A", "", None, "7"]:
    print(f"safe_int({v!r}) = {safe_int(v)}")

In [ ]:
# --- 2. Your own exception type, and raising it -------------------------
class LikertError(ValueError):
    """Raised when a value isn't a valid 1–5 Likert response."""

def clean_likert(n):
    """Return n if it's a valid 1–5 Likert int, else raise LikertError."""
    if isinstance(n, bool) or not isinstance(n, int):
        raise LikertError(f"{n!r} is not an integer")
    if not 1 <= n <= 5:
        raise LikertError(f"{n} is outside 1–5")
    return n

In [ ]:
# --- 3. clean a dirty survey column, keeping a rejection log ------------
raw_responses = ["5", "3", "N/A", "7", "", "1", "two", "4"]
clean, rejected = [], []
for r in raw_responses:
    n = safe_int(r)
    try:
        clean.append(clean_likert(n))        # may raise LikertError
    except LikertError as e:                 # catching the base ValueError works too
        rejected.append((r, str(e)))

print("\nclean:", clean)
print("rejected:")
for original, why in rejected:
    print(f"  {original!r}: {why}")

In [ ]:
# --- 4. Exception chaining: keep the original cause with `raise ... from` -
def parse_score(text):
    try:
        return int(text)
    except ValueError as e:
        raise LikertError(f"bad score {text!r}") from e   # preserves the __cause__

try:
    parse_score("oops")
except LikertError as e:
    print("\nraised:", e, "| caused by:", repr(e.__cause__))

In [ ]:
# --- 5. else / finally ---------------------------------------------------
def parse(value):
    try:
        n = int(value)
    except ValueError:
        return "bad"
    else:
        return f"ok:{n}"          # only when no exception
    finally:
        pass                       # cleanup would go here (e.g., close a file)

print("\n", parse("10"), parse("x"))

In [ ]:
# --- 6. assert: a cheap internal sanity check ---------------------------
def mean(xs):
    assert xs, "mean() needs at least one value"   # AssertionError if xs is empty
    return sum(xs) / len(xs)

print("mean:", mean([2, 4, 6]))

In [ ]:
# --- 7. TRAP: bare except hides real bugs (don't do this) ---------------
# try:
#     risky()
# except:            # ❌ catches EVERYTHING, even Ctrl+C and typos
#     pass           # ❌ and silently swallows the error
# Always: except SpecificError as e: ...

In [ ]:
# ======================================================================
# PART B — Files, Libraries & Research Data

In [ ]:
# ======================================================================

import csv
import json
import statistics
from collections import Counter
from pathlib import Path

HERE = Path(".")                 # notebook: files are written by the setup cell

In [ ]:
# --- 1. Read a CSV into a list of dicts ---------------------------------
with open(HERE / "students.csv", newline="") as f:
    students = list(csv.DictReader(f))   # each row is a dict keyed by header

print("rows read:", len(students))
print("first row:", students[0])

# CSV values are STRINGS — convert numbers (recall Session 1!)
scores = [int(s["score"]) for s in students]
print("class mean:", statistics.mean(scores))
print("class stdev:", round(statistics.stdev(scores), 2))
print("majors tally:", Counter(s["major"] for s in students))   # quick frequency count

In [ ]:
# --- 2. Group by major, mean score -------------------------------------
by_major = {}
for s in students:
    by_major.setdefault(s["major"], []).append(int(s["score"]))
major_means = {m: round(statistics.mean(v), 1) for m, v in by_major.items()}
print("means by major:", major_means)

In [ ]:
# --- 3. Write a summary CSV --------------------------------------------
with open(HERE / "students_summary.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["major", "mean_score", "n"])
    w.writeheader()
    for major, vals in by_major.items():
        w.writerow({"major": major, "mean_score": round(statistics.mean(vals), 1),
                    "n": len(vals)})
print("wrote students_summary.csv")

In [ ]:
# --- 4. JSON: serialize a Python object to text and back ----------------
snapshot = {"n": len(students), "mean": statistics.mean(scores), "by_major": major_means}
(HERE / "snapshot.json").write_text(json.dumps(snapshot, indent=2))   # pathlib write
restored = json.loads((HERE / "snapshot.json").read_text())           # pathlib read
print("\nJSON round-trip ok?", restored == snapshot)

In [ ]:
# --- 5. Survey: per-item means, skipping dirty values -------------------
def to_int(x):
    try:
        return int(x)
    except (ValueError, TypeError):
        return None        # handles "N/A" and "" (this session, Part A)

with open(HERE / "survey.csv", newline="") as f:
    rows = list(csv.DictReader(f))

items = [c for c in rows[0] if c != "respondent"]
item_means = {}
for item in items:
    vals = [to_int(r[item]) for r in rows]
    vals = [v for v in vals if v is not None]      # drop missing
    item_means[item] = round(statistics.mean(vals), 2)
print("\nper-item survey means:", item_means)

# Two files open at once in a single `with` (read source, write report together):
with open(HERE / "survey.csv", newline="") as src, \
     open(HERE / "survey_summary.csv", "w", newline="") as out:
    rows = list(csv.DictReader(src))
    w = csv.DictWriter(out, fieldnames=["item", "mean", "n_valid"])
    w.writeheader()
    for item in items:
        valid = [to_int(r[item]) for r in rows if to_int(r[item]) is not None]
        w.writerow({"item": item, "mean": round(statistics.mean(valid), 2),
                    "n_valid": len(valid)})
print("wrote survey_summary.csv")

In [ ]:
# --- 6. pathlib: discover files without hard-coding names ---------------
csvs = sorted(p.stem for p in HERE.glob("*.csv"))   # .stem = filename without extension
print("\nCSV files here:", csvs)

## Now you try — practice

# Session 4 — Practice: Exceptions, Files & Research Data

This 2-hour session has two halves. Do **Part A** after the first topic, **Part B** after the second. Predict every output before you run it.

## Part A — Exceptions & Defensive Code

### Task 1 — `safe_int`
Write `safe_int(value)` returning `int(value)` or `None` on failure. Test on
`"42"`, `"N/A"`, `""`, `None`, `3.0`.

### Task 2 — Clean a survey column
Given `raw = ["5","3","N/A","7","","1","two","4"]`, produce:
- `clean` — list of valid Likert ints (1–5), and
- `rejected` — list of `(value, reason)` pairs.
Use a `clean_likert(n)` that **raises** `ValueError` for out-of-range or non-ints.

### Task 3 — Write a test
Put `clean_likert` in `clean.py` and write `test_clean.py` with pytest:
one passing case and one `pytest.raises(ValueError)` case. Run `pytest`.

### Task 4 — Discuss
Why is `except:` (bare) dangerous? Give one error it would hide that you'd rather see.

### Bonus — Pythonic idiom drill
Cover the `# ->` answers, predict each line, then run.

```python
class LikertError(ValueError):       # your own exception type
    pass
print(issubclass(LikertError, ValueError))   # -> True  (so `except ValueError` still catches it)

try:
    assert 1 == 2, "values differ"   # assert: cheap internal sanity check
except AssertionError as e:
    print(e)                         # -> values differ
```

## Part B — Files, Libraries & Research Data

Files provided: `students.csv`, `survey.csv`.

### Task 1 — Read & summarize students
Read `students.csv` with `csv.DictReader`. Remember the values are **strings** — convert
`score` to `int`. Print the class mean and median with the `statistics` module.

### Task 2 — Mean by major
Build `{major: mean_score}`. (Hint: `dict.setdefault(key, []).append(...)`.)

### Task 3 — Clean & summarize the survey
`survey.csv` has `"N/A"` and blanks in numeric columns. For each `q*` item, compute the
mean of the **valid** values only, and how many were valid. Write `survey_summary.csv`
with columns `item,mean,n_valid`.

### Task 4 — pandas teaser (optional)
If `pandas` is installed: `pd.read_csv("students.csv")["score"].describe()`. Compare the
mean to your hand-computed one.

### Trap check
What happens if you accidentally open `students.csv` with mode `"w"` before reading it?

### Bonus — Pythonic idiom drill
Cover the `# ->` answers, predict each line, then run.

```python
import json
s = json.dumps({"n": 3, "ok": True})
print(s)                             # -> {"n": 3, "ok": true}   (Python True -> JSON true)
print(json.loads(s)["ok"])           # -> True                   (and back to a Python bool)
```

---

In [ ]:
# Your practice work — type here. Predict before you run.


<details>
<summary><strong>Show solutions</strong></summary>

### Part A — Exceptions & Defensive Code

```python
def safe_int(value):
    try:
        return int(value)
    except (ValueError, TypeError):
        return None

def clean_likert(n):
    if isinstance(n, bool) or not isinstance(n, int):
        raise ValueError(f"{n!r} not an int")
    if not 1 <= n <= 5:
        raise ValueError(f"{n} outside 1–5")
    return n

raw = ["5","3","N/A","7","","1","two","4"]
clean, rejected = [], []
for r in raw:
    try:
        clean.append(clean_likert(safe_int(r)))
    except ValueError as e:
        rejected.append((r, str(e)))
print(clean)      # [5, 3, 1, 4]
print(rejected)   # [('N/A', ...), ('7', ...), ('', ...), ('two', ...)]
```

```python
# test_clean.py
import pytest
from clean import clean_likert
def test_ok():   assert clean_likert(3) == 3
def test_bad():
    with pytest.raises(ValueError):
        clean_likert(9)
```

Task 4: a bare `except:` also catches `KeyboardInterrupt` (Ctrl+C) and `NameError`
from your own typos — so a misspelled variable would be silently swallowed instead of
showing you the bug. Always catch the specific exception you expect.

### Part B — Files, Libraries & Research Data

See `demo.py` in this folder — it implements Tasks 1–3 exactly. Key lines:

```python
scores = [int(s["score"]) for s in students]     # convert strings!
statistics.mean(scores)                           # 75.5

by_major = {}
for s in students:
    by_major.setdefault(s["major"], []).append(int(s["score"]))

def to_int(x):
    try: return int(x)
    except (ValueError, TypeError): return None   # handles N/A and ""
```

Trap: opening with `"w"` **truncates the file to empty immediately** — your data is gone
before you ever read it. Use `"r"` (the default) to read.

</details>